<a href="https://colab.research.google.com/github/Vyankatesh-ops/Advanced-Machine-Learning/blob/main/Experiment_8_AML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch nltk numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
INPUT_SIZE = 5000   # vocab size
EMBED_SIZE = 128
HIDDEN_SIZE = 256
OUTPUT_SIZE = 5000

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(INPUT_SIZE, EMBED_SIZE)
        self.lstm = nn.LSTM(EMBED_SIZE, HIDDEN_SIZE, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(x)
        return hidden, cell

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(OUTPUT_SIZE, EMBED_SIZE)
        self.lstm = nn.LSTM(EMBED_SIZE, HIDDEN_SIZE, batch_first=True)
        self.fc = nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE)

    def forward(self, x, hidden, cell):
        x = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(x, (hidden, cell))
        predictions = self.fc(outputs)
        return predictions, hidden, cell


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target):
        hidden, cell = self.encoder(source)
        outputs, _, _ = self.decoder(target, hidden, cell)
        return outputs

In [ ]:
encoder = Encoder()
decoder = Decoder()
model = Seq2Seq(encoder, decoder)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model Ready!")

Model Ready!


In [ ]:
# Example random input (replace with real tokenized data)
source = torch.randint(0, INPUT_SIZE, (32, 50))   # batch, seq_len
target = torch.randint(0, OUTPUT_SIZE, (32, 20))

# Training step
optimizer.zero_grad()
output = model(source, target)

loss = criterion(output.view(-1, OUTPUT_SIZE), target.view(-1))
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 8.518838882446289


In [ ]:
# Example input (replace with real tokenized document)
source = torch.randint(0, INPUT_SIZE, (1, 50))  # (batch_size, seq_len)

# Step 1: Get hidden state from encoder
hidden, cell = encoder(source)

# Step 2: Start decoding
input_word = torch.tensor([[1]])  # START token
max_len = 20

generated_summary = []

for _ in range(max_len):
    output, hidden, cell = decoder(input_word, hidden, cell)

    next_word = output.argmax(2).item()
    generated_summary.append(next_word)

    input_word = torch.tensor([[next_word]])

print("Generated Summary Tokens:", generated_summary)

Generated Summary Tokens: [865, 1425, 1425, 4411, 75, 4689, 2877, 4486, 2929, 1089, 3139, 402, 2310, 3930, 3285, 2572, 1029, 3309, 1067, 2891]
